In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np
import json
import os
import time
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from tqdm import tqdm

In [2]:
BATCH_SIZE = 128
EPOCHS = 20
LR = 0.001
WEIGHT_DECAY = 0.0001
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SAVE_DIR = "results_vgg_cifar"
os.makedirs(SAVE_DIR, exist_ok=True)

torch.manual_seed(42)
np.random.seed(42)

In [3]:
# Статистики для CIFAR-10
MEAN = (0.4914, 0.4822, 0.4465)
STD = (0.2471, 0.2435, 0.2616)

transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD)
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD)
])

train_set = torchvision.datasets.CIFAR10(root="./data", train=True, download=True, transform=transform_train)
test_set = torchvision.datasets.CIFAR10(root="./data", train=False, download=True, transform=transform_test)

trainloader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True)
testloader = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)


100%|██████████| 170M/170M [00:10<00:00, 16.6MB/s]
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


In [4]:
import torchvision.models as models

model_vgg_original = models.vgg16(pretrained=True)
print(model_vgg_original)

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth


100%|██████████| 528M/528M [00:07<00:00, 69.6MB/s]


VGG(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU(inplace=True)
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (6): ReLU(inplace=True)
    (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): ReLU(inplace=True)
    (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): ReLU(inplace=True)
    (14): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (15): ReLU(inplace=True)
    (16): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1

In [5]:
class ModifiedVGG(nn.Module):
    #отличия от VGG16 - 3 блока свертки вместо 5 для работы с 28*28, изображения чб 28*28,
    #AdaptiveAvgPool2d перед классификатором для стабильной размерности, BatchNorm + Dropout для регуляризации
    def __init__(self, num_classes=10, dropout_rate=0.5):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),  # (3, 32, 32) -> (64, 16, 16)

            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),  # (64, 16, 16) -> (128, 8, 8)

            nn.Conv2d(128, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),  # (128, 8, 8) -> (256, 4, 4)
        )

        self.adaptive_pool = nn.AdaptiveAvgPool2d((4, 4)) #(256, 4, 4) -> (256, 4, 4)

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 4 * 4, 1024), nn.BatchNorm1d(1024), nn.ReLU(inplace=True),
            nn.Dropout(dropout_rate),
            nn.Linear(1024, 512), nn.BatchNorm1d(512), nn.ReLU(inplace=True),
            nn.Dropout(dropout_rate),
            nn.Linear(512, num_classes)
        )
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
            elif isinstance(m, (nn.BatchNorm2d, nn.BatchNorm1d)):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, x):
        x = self.features(x)
        x = self.adaptive_pool(x)
        return self.classifier(x)

model = ModifiedVGG().to(DEVICE)
print(f"Параметры модели: {sum(p.numel() for p in model.parameters()):,}")

Параметры модели: 6,466,122


In [6]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

train_losses, val_losses, train_accs, val_accs = [], [], [], []
best_val_acc = 0.0
start_time = time.time()

for epoch in range(EPOCHS):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    pbar = tqdm(trainloader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]", leave=False)

    for inputs, labels in pbar:
        inputs, labels = inputs.to(DEVICE, non_blocking=True), labels.to(DEVICE, non_blocking=True)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
        pbar.set_postfix(loss=f"{loss.item():.4f}")

    train_losses.append(running_loss / len(trainloader))
    train_accs.append(100.0 * correct / total)

    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    with torch.no_grad():
        for inputs, labels in testloader:
            inputs, labels = inputs.to(DEVICE, non_blocking=True), labels.to(DEVICE, non_blocking=True)
            outputs = model(inputs)
            val_loss += criterion(outputs, labels).item()
            _, predicted = outputs.max(1)
            val_total += labels.size(0)
            val_correct += predicted.eq(labels).sum().item()

    val_losses.append(val_loss / len(testloader))
    val_accs.append(100.0 * val_correct / val_total)

    current_val_acc = val_accs[-1]
    if current_val_acc > best_val_acc:
        best_val_acc = current_val_acc
        torch.save(model.state_dict(), os.path.join(SAVE_DIR, "best_model.pth"))

    print(f"Epoch {epoch+1:02d} | Train Loss: {train_losses[-1]:.4f} Acc: {train_accs[-1]:.2f}% | "
          f"Val Loss: {val_losses[-1]:.4f} Acc: {current_val_acc:.2f}% | Best: {best_val_acc:.2f}%")

train_time = time.time() - start_time
print(f"Обучение завершено за {train_time:.2f} сек. | Лучшая точность валидации: {best_val_acc:.2f}%")


Epoch 1/20 [Train]:   0%|          | 0/391 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


Epoch 01 | Train Loss: 1.5327 Acc: 46.59% | Val Loss: 1.1989 Acc: 56.87% | Best: 56.87%


Epoch 02 | Train Loss: 0.9449 Acc: 67.00% | Val Loss: 0.7760 Acc: 72.72% | Best: 72.72%


Epoch 03 | Train Loss: 0.7442 Acc: 74.30% | Val Loss: 0.7825 Acc: 74.46% | Best: 74.46%


Epoch 04 | Train Loss: 0.6385 Acc: 78.04% | Val Loss: 0.6902 Acc: 77.23% | Best: 77.23%


...

Epoch 16 | Train Loss: 0.2307 Acc: 92.04% | Val Loss: 0.3690 Acc: 88.19% | Best: 88.88%


Epoch 17 | Train Loss: 0.2183 Acc: 92.51% | Val Loss: 0.3800 Acc: 88.07% | Best: 88.88%


Epoch 18 | Train Loss: 0.2073 Acc: 92.89% | Val Loss: 0.3296 Acc: 89.52% | Best: 89.52%


Epoch 19 | Train Loss: 0.1956 Acc: 93.29% | Val Loss: 0.3415 Acc: 89.26% | Best: 89.52%


Epoch 20 | Train Loss: 0.1837 Acc: 93.68% | Val Loss: 0.3474 Acc: 89.50% | Best: 89.52%
Обучение завершено за 487.99 сек. | Лучшая точность валидации: 89.52%


In [7]:
#finally
model.load_state_dict(torch.load(os.path.join(SAVE_DIR, "best_model.pth"), map_location=DEVICE))
model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for inputs, labels in testloader:
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        outputs = model(inputs)
        _, predicted = outputs.max(1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']

report = classification_report(all_labels, all_preds, target_names=class_names, output_dict=True, zero_division=0)
cm = confusion_matrix(all_labels, all_preds)

print("\nClassification Report:")
print(classification_report(all_labels, all_preds, target_names=class_names))

metrics = {
    "train_time_sec": f"{train_time:.2f}",
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "lr": LR,
    "weight_decay": WEIGHT_DECAY,
    "best_val_accuracy": f"{best_val_acc:.4f}",
    "final_test_accuracy": f"{report['accuracy']:.4f}",
    "macro_avg_f1": f"{report['macro avg']['f1-score']:.4f}",
    "weighted_avg_f1": f"{report['weighted avg']['f1-score']:.4f}"
}
with open(os.path.join(SAVE_DIR, "metrics.json"), "w") as f:
    json.dump(metrics, f, indent=4)


Classification Report:
              precision    recall  f1-score   support

    airplane       0.88      0.92      0.90      1000
  automobile       0.95      0.95      0.95      1000
        bird       0.86      0.86      0.86      1000
         cat       0.84      0.72      0.78      1000
        deer       0.88      0.89      0.89      1000
         dog       0.84      0.84      0.84      1000
        frog       0.91      0.93      0.92      1000
       horse       0.91      0.94      0.92      1000
        ship       0.94      0.95      0.95      1000
       truck       0.93      0.95      0.94      1000

    accuracy                           0.90     10000
   macro avg       0.89      0.90      0.89     10000
weighted avg       0.89      0.90      0.89     10000



In [8]:
#Visualize it
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(train_losses, label="Train Loss"); axes[0].plot(val_losses, label="Val Loss")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss"); axes[0].legend(); axes[0].grid()
axes[1].plot(train_accs, label="Train Acc"); axes[1].plot(val_accs, label="Val Acc")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Accuracy (%)"); axes[1].legend(); axes[1].grid()
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "learning_curves.png"), dpi=150)
plt.close()

disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
fig, ax = plt.subplots(figsize=(10, 8))
disp.plot(cmap=plt.cm.Blues, ax=ax, xticks_rotation=45, values_format=".0f")
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "confusion_matrix.png"), dpi=150)
plt.close()
